# API Deployment with FastAPI

Serving ML models via REST APIs is the most common deployment pattern. This notebook covers:
1. **Saving and loading** a trained sklearn model
2. **Building a FastAPI endpoint** for predictions
3. **Testing** the API with `requests`
4. **Input validation** with Pydantic schemas

In [ ]:
import numpy as np
import pandas as pd
import json, os, pickle, tempfile
from sklearn.datasets import load_iris
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

try:
    from fastapi import FastAPI
    from pydantic import BaseModel
    import uvicorn
    HAS_FASTAPI = True
except ImportError:
    HAS_FASTAPI = False
    print('fastapi/uvicorn not installed -- pip install fastapi uvicorn')

print('Setup complete.')

## 1. Train and Save a Model

We train a simple classifier and serialise it with `pickle` (or `joblib`).
In production you would use MLflow or a model registry.

In [ ]:
# Train
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

model = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
model.fit(X_train, y_train)
print(f'Test accuracy: {accuracy_score(y_test, model.predict(X_test)):.4f}')

# Save
MODEL_PATH = os.path.join(tempfile.gettempdir(), 'iris_model.pkl')
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model, f)
print(f'Model saved to {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024:.1f} KB)')

## 2. Define the FastAPI Application

Key components:
- **Pydantic model** for request validation
- **Startup event** to load the model once
- **`/predict`** endpoint returning class and probabilities

In [ ]:
# We write the FastAPI app to a file so it can be run with uvicorn
APP_CODE = f'''
import pickle, numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List

# ---------- Pydantic schemas ----------
class PredictionRequest(BaseModel):
    features: List[float] = Field(..., min_length=4, max_length=4,
                                   description="4 Iris features: sepal_length, sepal_width, petal_length, petal_width")

class PredictionResponse(BaseModel):
    predicted_class: int
    class_name: str
    probabilities: List[float]

# ---------- App ----------
app = FastAPI(title="Iris Classifier API", version="1.0")

CLASS_NAMES = ["setosa", "versicolor", "virginica"]
MODEL = None

@app.on_event("startup")
def load_model():
    global MODEL
    with open("{MODEL_PATH}", "rb") as f:
        MODEL = pickle.load(f)

@app.get("/health")
def health():
    return {{"status": "ok", "model_loaded": MODEL is not None}}

@app.post("/predict", response_model=PredictionResponse)
def predict(req: PredictionRequest):
    X = np.array(req.features).reshape(1, -1)
    pred = int(MODEL.predict(X)[0])
    proba = MODEL.predict_proba(X)[0].tolist()
    return PredictionResponse(
        predicted_class=pred,
        class_name=CLASS_NAMES[pred],
        probabilities=[round(p, 4) for p in proba],
    )
'''

app_path = os.path.join(tempfile.gettempdir(), 'iris_api.py')
with open(app_path, 'w') as f:
    f.write(APP_CODE)

print(f'FastAPI app written to {app_path}')
print('Run with: uvicorn iris_api:app --reload')
print(APP_CODE)

## 3. Launch the Server (background)

In a real setting you would run `uvicorn iris_api:app --host 0.0.0.0 --port 8000`.
Here we start it in a background thread for demonstration.

In [ ]:
import threading, time, importlib, sys

if HAS_FASTAPI:
    # Add temp dir to path so the module can be imported
    sys.path.insert(0, tempfile.gettempdir())
    import iris_api
    importlib.reload(iris_api)

    config = uvicorn.Config(iris_api.app, host='127.0.0.1', port=8000, log_level='warning')
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    time.sleep(2)  # give server time to start
    print('Server running on http://127.0.0.1:8000')
else:
    print('Skipping server launch (fastapi not available).')

## 4. Test the API with `requests`

In [ ]:
import requests

BASE = 'http://127.0.0.1:8000'

if HAS_FASTAPI:
    # Health check
    r = requests.get(f'{BASE}/health')
    print('Health:', r.json())

    # Single prediction
    payload = {'features': [5.1, 3.5, 1.4, 0.2]}
    r = requests.post(f'{BASE}/predict', json=payload)
    print('Prediction:', r.json())

    # Batch test -- send several samples
    samples = [
        [6.7, 3.1, 4.7, 1.5],  # versicolor
        [7.7, 3.0, 6.1, 2.3],  # virginica
        [4.9, 3.0, 1.4, 0.2],  # setosa
    ]
    for s in samples:
        r = requests.post(f'{BASE}/predict', json={'features': s})
        print(f'  {s} -> {r.json()["class_name"]} (p={max(r.json()["probabilities"]):.3f})')
else:
    # Simulate the prediction locally
    with open(MODEL_PATH, 'rb') as f:
        loaded = pickle.load(f)
    sample = np.array([[5.1, 3.5, 1.4, 0.2]])
    print('Local prediction:', iris.target_names[loaded.predict(sample)[0]])

## 5. Dockerising the API

A minimal `Dockerfile` for production:

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "iris_api:app", "--host", "0.0.0.0", "--port", "8000"]
```

**Best practices:**
- Pin dependency versions in `requirements.txt`
- Use multi-stage builds for smaller images
- Add a `/health` endpoint for orchestrator liveness probes
- Set resource limits (CPU/memory) in Kubernetes or Docker Compose

## Key Takeaways

- **FastAPI** provides automatic OpenAPI docs, request validation, and async support.
- Always validate inputs with **Pydantic** schemas.
- Load the model **once at startup**, not per request.
- Add a `/health` endpoint for monitoring and orchestration.
- **Containerise** with Docker for reproducible deployments.

**Next:** Monitoring model performance and data drift in production.